# Klasyfikacja obiektów trudnych.

## Cel ćwiczenia

Wykorzystując zdobytą dotychczas wiedzę doprowadź do rozpoznania i separacji (wydzielenia) obiektów widocznych na obrazie `details.png`.

Obiekty należą do trzech różnych klas: **śrubki**, **nakrętki** oraz **podkładki**.

Rozpoznanie powinno być dokonane na podstawie **momentów**, po wcześniejszej wstępnej obróbce obrazu oraz binaryzacji (segmentacji) - w miarę możliwości należy wyeliminować różnice w oświetleniu sceny, odblaski oraz cienie rzucane przez obiekty.

Końcowym rezultatem zadania mają być **trzy** obrazy zawierające tylko obiekty należące do danej klasy (osobno śrubki, osobno nakrętki i osobno podkładki).


In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np

if not os.path.exists("details.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/13_CCL/details.png --no-check-certificate


im = cv2.imread( 'details.png', cv2.IMREAD_GRAYSCALE)

plt.imshow( im, 'gray')
plt.axis('off')
plt.show()

# Implementacja procesu analizy i klasyfikacji

In [ ]:
im = cv2.imread("details.png", cv2.IMREAD_GRAYSCALE)

plt.figure(figsize=(6, 4))
plt.imshow(im, cmap="gray")
plt.axis("off")
plt.title("details.png")
plt.show()

blur = cv2.GaussianBlur(im, (5, 5), 1.2)
edges = cv2.Canny(blur, 40, 120)
bin_img = cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=1)

contours, hierarchy = cv2.findContours(bin_img, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
hierarchy = hierarchy[0]

debug = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)

screws = np.zeros_like(im)
nuts = np.zeros_like(im)
washers = np.zeros_like(im)

obj_id = 1

print("id | hu1 | circularity | fill_ratio | klasa")

for i, c in enumerate(contours):
    if hierarchy[i][3] != -1:
        continue

    area = cv2.contourArea(c)
    if area < 250:
        continue

    mask = np.zeros_like(im)
    cv2.drawContours(mask, [c], -1, 255, -1)

    hu = cv2.HuMoments(cv2.moments(mask)).flatten()
    hu1 = -np.sign(hu[0]) * np.log10(np.abs(hu[0]) + 1e-30)

    perimeter = cv2.arcLength(c, True)
    circularity = 4 * np.pi * area / (perimeter ** 2)

    (_, _), r = cv2.minEnclosingCircle(c)
    fill_ratio = area / (np.pi * r * r)

    if hu1 < 3.12 or circularity < 0.65:
        cls = "screw"
        screws[mask > 0] = im[mask > 0]
        color = (0, 0, 255)

    elif fill_ratio > 0.85:
        cls = "washer"
        washers[mask > 0] = im[mask > 0]
        color = (0, 255, 0)

    else:
        cls = "nut"
        nuts[mask > 0] = im[mask > 0]
        color = (255, 0, 0)

    M = cv2.moments(c)
    cx = int(M["m10"] / M["m00"])
    cy = int(M["m01"] / M["m00"])

    cv2.putText(debug, f"{obj_id}", (cx - 8, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    print(f"{obj_id:2d} | {hu1:.3f} | {circularity:.3f} | {fill_ratio:.3f} | {cls}")

    obj_id += 1

plt.figure(figsize=(16, 8))

plt.subplot(2, 3, 1)
plt.title("Oryginal")
plt.imshow(im, cmap="gray")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.title("Segmentacja")
plt.imshow(bin_img, cmap="gray")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.title("Obiekty ponumerowane")
plt.imshow(cv2.cvtColor(debug, cv2.COLOR_BGR2RGB))
plt.axis("off")

plt.subplot(2, 3, 4)
plt.title("Srubki")
plt.imshow(screws, cmap="gray")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.title("Nakretki")
plt.imshow(nuts, cmap="gray")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.title("Podkladki")
plt.imshow(washers, cmap="gray")
plt.axis("off")

plt.tight_layout()
plt.show()
